# BFSI Customer Onboarding - Exploratory Data Analysis

**Author:** Sagar Kandelkar  
**Date:** September 2026  
**Dataset:** `customer_applications.csv` — 15 sample onboarding applications

## Objectives
1. Understand the distribution of applications across channels, products, and regions
2. Analyze risk profiles and credit score patterns
3. Identify bottlenecks in the onboarding pipeline
4. Derive actionable insights for process improvement

In [ ]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"Python: {pd.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
# ============================================================
# 2. DATA LOADING & OVERVIEW
# ============================================================

# Load the dataset
df = pd.read_csv('../data/customer_applications.csv')

# Basic info
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn Names:\n{list(df.columns)}")

# Display first few rows
df.head()

In [ ]:
# ============================================================
# 3. DATA PROFILING
# ============================================================

print("=== DATA TYPES & MISSING VALUES ===")
print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
print(f"\n=== DUPLICATE ROWS: {df.duplicated().sum()} ===")

# Summary statistics for numeric columns
numeric_cols = ['annual_income', 'credit_score']
print("\n=== NUMERIC SUMMARY ===")
df[numeric_cols].describe().round(2)

## 4. Channel & Product Analysis

In [ ]:
# Channel distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Channel pie chart
channel_counts = df['application_channel'].value_counts()
colors_channel = ['#1a5fb4', '#26a269']
axes[0].pie(channel_counts.values, labels=channel_counts.index, autopct='%1.1f%%', 
            colors=colors_channel, startangle=90, explode=[0.02, 0.02])
axes[0].set_title('Applications by Channel', fontsize=14, fontweight='bold')

# Product bar chart
product_counts = df['product_type'].value_counts()
colors_product = ['#1a5fb4', '#26a269', '#e5a50a', '#c01c28']
bars = axes[1].bar(range(len(product_counts)), product_counts.values, color=colors_product)
axes[1].set_xticks(range(len(product_counts)))
axes[1].set_xticklabels(product_counts.index, rotation=15, ha='right')
axes[1].set_title('Applications by Product Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nChannel Distribution:")
print(channel_counts)
print(f"\nDigital Adoption Rate: {channel_counts['Digital']/len(df)*100:.1f}%")

## 5. Geographic Distribution

In [ ]:
# State-wise distribution
state_counts = df['state'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.8, len(state_counts)))
bars = ax.barh(state_counts.index[::-1], state_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('Number of Applications')
ax.set_title('Applications by State', fontsize=14, fontweight='bold')

# Add value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2.,
           f' {int(width)}', ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("State-wise Breakdown:")
print(state_counts)

## 6. Onboarding Pipeline & Status Analysis

In [ ]:
# Status distribution
status_counts = df['onboarding_status'].value_counts()
status_colors = {'Completed': '#26a269', 'In Progress': '#e5a50a', 
                 'On Hold': '#c01c28', 'Rejected': '#9c5c5c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Status donut chart
status_colors_list = [status_colors.get(s, '#999') for s in status_counts.index]
wedges, texts, autotexts = axes[0].pie(status_counts.values, labels=status_counts.index, 
                                        autopct='%1.1f%%', colors=status_colors_list,
                                        startangle=90, pctdistance=0.75)
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
axes[0].add_artist(centre_circle)
axes[0].set_title('Onboarding Status Distribution', fontsize=14, fontweight='bold')

# Status by channel crosstab
crosstab = pd.crosstab(df['application_channel'], df['onboarding_status'])
crosstab.plot(kind='bar', ax=axes[1], color=[status_colors.get(c, '#999') for c in crosstab.columns])
axes[1].set_title('Status by Channel', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Channel')
axes[1].set_ylabel('Count')
axes[1].legend(title='Status', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# Conversion metrics
completed = status_counts.get('Completed', 0)
total = len(df)
print(f"\n=== PIPELINE METRICS ===")
print(f"Total Applications: {total}")
print(f"Completed: {completed} ({completed/total*100:.1f}%)")
print(f"In Progress: {status_counts.get('In Progress', 0)}")
print(f"On Hold: {status_counts.get('On Hold', 0)}")
print(f"Rejected: {status_counts.get('Rejected', 0)}")
print(f"Conversion Rate: {completed/total*100:.1f}%")

## 7. Risk & Credit Score Analysis

In [ ]:
# Credit score distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of credit scores
axes[0, 0].hist(df['credit_score'], bins=8, color='#1a5fb4', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(df['credit_score'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["credit_score"].mean():.0f}')
axes[0, 0].set_xlabel('Credit Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Credit Score Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()

# AML Risk distribution
aml_counts = df['aml_risk_score'].value_counts()
aml_colors = {'Low': '#26a269', 'Medium': '#e5a50a', 'High': '#c01c28'}
aml_color_list = [aml_colors.get(r, '#999') for r in aml_counts.index]
axes[0, 1].pie(aml_counts.values, labels=aml_counts.index, autopct='%1.1f%%', 
               colors=aml_color_list, startangle=90)
axes[0, 1].set_title('AML Risk Distribution', fontsize=12, fontweight='bold')

# Credit Score vs AML Risk (Box plot)
sns.boxplot(data=df, x='aml_risk_score', y='credit_score', 
            order=['Low', 'Medium', 'High'],
            palette=['#26a269', '#e5a50a', '#c01c28'], ax=axes[1, 0])
axes[1, 0].set_title('Credit Score by AML Risk', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('AML Risk Score')

# Income vs Credit Score scatter
risk_color_map = {'Low': '#26a269', 'Medium': '#e5a50a', 'High': '#c01c28'}
for risk in df['aml_risk_score'].unique():
    subset = df[df['aml_risk_score'] == risk]
    axes[1, 1].scatter(subset['annual_income']/100000, subset['credit_score'], 
                      label=f'{risk} Risk', color=risk_color_map.get(risk, '#999'), s=100, alpha=0.7)
axes[1, 1].set_xlabel('Annual Income (₹ Lakhs)')
axes[1, 1].set_ylabel('Credit Score')
axes[1, 1].set_title('Income vs Credit Score by Risk', fontsize=12, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n=== RISK PROFILE SUMMARY ===")
print(df.groupby('aml_risk_score').agg({
    'credit_score': ['mean', 'min', 'max'],
    'annual_income': 'mean'
}).round(2))

## 8. Employment & Income Analysis

In [ ]:
# Employment type analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Employment type distribution
emp_counts = df['employment_type'].value_counts()
axes[0].bar(emp_counts.index, emp_counts.values, color=['#1a5fb4', '#26a269', '#e5a50a', '#c01c28'][:len(emp_counts)])
axes[0].set_title('Applications by Employment Type', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(emp_counts.values):
    axes[0].text(i, v, f' {v}', ha='center', va='bottom', fontweight='bold')

# Income by employment type (violin plot)
sns.violinplot(data=df, x='employment_type', y='annual_income', ax=axes[1], palette='Blues')
axes[1].set_title('Income Distribution by Employment', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Annual Income (₹)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'₹{x/100000:.0f}L'))

plt.tight_layout()
plt.show()

print("\nIncome by Employment Type:")
print(df.groupby('employment_type')['annual_income'].agg(['count', 'mean', 'median']).round(0))

## 9. KYC Compliance Analysis

In [ ]:
# KYC Status analysis
fig, ax = plt.subplots(figsize=(10, 5))

kyc_counts = df['kyc_status'].value_counts()
bars = ax.bar(kyc_counts.index, kyc_counts.values, color=['#26a269', '#e5a50a'])
ax.set_title('KYC Verification Status', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# Add compliance rate annotation
verified = kyc_counts.get('Verified', 0)
total = len(df)
compliance_rate = verified / total * 100
ax.axhline(y=verified, color='green', linestyle='--', alpha=0.5)
ax.text(0.5, verified + 0.3, f'Compliance Rate: {compliance_rate:.1f}%', 
        ha='center', fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

# KYC vs Onboarding Status crosstab
print("\nKYC Status vs Onboarding Status:")
print(pd.crosstab(df['kyc_status'], df['onboarding_status'], margins=True))

## 10. Correlation & Insights

In [ ]:
# Correlation matrix for numeric variables
numeric_df = df[['annual_income', 'credit_score']].copy()

# Add encoded versions for correlation
numeric_df['aml_risk_encoded'] = df['aml_risk_score'].map({'Low': 1, 'Medium': 2, 'High': 3})
numeric_df['status_encoded'] = df['onboarding_status'].map({'Completed': 4, 'In Progress': 3, 'On Hold': 2, 'Rejected': 1})

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
            square=True, fmt='.2f', ax=ax,
            xticklabels=['Income', 'Credit Score', 'AML Risk', 'Status'],
            yticklabels=['Income', 'Credit Score', 'AML Risk', 'Status'])
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== KEY INSIGHTS ===")
print(f"1. Digital channel accounts for {df[df['application_channel']=='Digital'].shape[0]/len(df)*100:.0f}% of applications")
print(f"2. Average credit score: {df['credit_score'].mean():.0f}")
print(f"3. {df[df['aml_risk_score']=='Low'].shape[0]} applications ({df[df['aml_risk_score']=='Low'].shape[0]/len(df)*100:.0f}%) are low-risk and eligible for auto-approval")
print(f"4. Onboarding completion rate: {df[df['onboarding_status']=='Completed'].shape[0]/len(df)*100:.1f}%")
print(f"5. Average income: ₹{df['annual_income'].mean()/100000:.1f} Lakhs")
print(f"6. Rejection rate: {df[df['onboarding_status']=='Rejected'].shape[0]/len(df)*100:.1f}%")

## 11. Export Summary Statistics

In [ ]:
# Generate summary report
summary = {
    'Total Applications': len(df),
    'Digital Channel %': round(df[df['application_channel']=='Digital'].shape[0]/len(df)*100, 1),
    'Avg Credit Score': round(df['credit_score'].mean(), 0),
    'Avg Income (₹)': round(df['annual_income'].mean(), 0),
    'Completion Rate %': round(df[df['onboarding_status']=='Completed'].shape[0]/len(df)*100, 1),
    'Rejection Rate %': round(df[df['onboarding_status']=='Rejected'].shape[0]/len(df)*100, 1),
    'Low Risk %': round(df[df['aml_risk_score']=='Low'].shape[0]/len(df)*100, 1),
    'KYC Verified %': round(df[df['kyc_status']=='Verified'].shape[0]/len(df)*100, 1)
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("=== EXECUTIVE SUMMARY ===")
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv('../data/summary_statistics.csv', index=False)
print("\nSummary saved to: summary_statistics.csv")

---

**Conclusion:**

This EDA reveals several actionable insights for the BFSI onboarding project:

1. **Digital adoption is strong** (60%) but there's room to improve the branch-to-digital conversion
2. **Low-risk applications (60%)** can be auto-approved, reducing manual workload
3. **Credit scores correlate with AML risk** — higher scores generally indicate lower risk
4. **Completion rate of 46.7%** suggests bottlenecks in the later stages of onboarding
5. **KYC compliance at 80%** — pending verifications need faster turnaround

These findings support the TO-BE recommendations for automation and real-time API integration.